# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: d:\Documents\An-2\08-Ingineria-AI\echochamber-project-team-2
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_02"
model = "gemini-2.5-flash-lite"
temperature = 0.6
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [5]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [6]:
top_15 = df['source_channel'].value_counts().head(15)


In [7]:
# cele mai frecvente 15 canale sursă din dataset

print(top_15)
 # completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64


In [8]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [9]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


In [19]:
sample_df2 = df.sample(10, random_state=99).copy()
sample_df2[["source_channel", "text"]]

,source_channel,text
25609,turcescu111,posibil chiar Rusia sa il fi sponsorizat pe Ha...
25012,turcescu111,"Cand l-am vazut pe Muc cu ,,picioarele-n 'ante..."
9081,RecorderRomania,Din ce aud eu tu puteai să te pensionezi. Ce t...
1851,georgesimionoficial,Doamne ajuta Simion-Georgescu! Felicitari moro...
384,georgesimionoficial,Nu o sa le tina mult! Mr Trump nu ii place pe ...
3887,digi24hd56,Americanii se bazeaza pe Peteskyan actualul pr...
12985,RecorderRomania,Cel mai tulburator documentar (probabil printr...
673,georgesimionoficial,Ferească Dumnezeu 😮 Nu mai putem deschide gura...
21256,CălinGeorgescu-CanalulOficial,Multumim pt mesaj! 🙏 Trebuie CLAR boicotat vot...
20947,CălinGeorgescu-CanalulOficial,"Este dreptul nostru legal,constituțional-democ..."


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [ ]:
# IMPORTANT - SCHIMBA PROMTUL DE MAI JOS PENTRU A SE POTRIVI CU CERINȚELE TALE ȘI ASIGURĂ-TE CĂ RESPECTĂ STRUCTURA SOLICITATĂ
# include în prompt instrucțiuni clare pentru fiecare dintre cele 7 elemente pe care vrei să le extragi și asigură-te că modelul înțelege că trebuie să returneze un JSON valid cu exact acele chei
# inlocueste "..." cu instrucțiuni clare pentru fiecare element
# Prompt de sistem: definește rolul modelului
# poti pune si alte axe de analiza care te intereseaza


SYSTEM_PROMPT = """
Ești un analist specializat în analiza discursului politic din Romania. 
Rolul tău este să analizezi comentarii politice și să extragi informații structurate din comentarii, prin contextualizare in raport cu titlul stirilor.
Trebuie să fii obiectiv, să eviți bias-ul personal și să te bazezi pe textul comentariului.
Raspunsurile se vor returna in JSON valid.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și identifică:
1. target: Persoana, partidul, instituția sau grupul vizat de critică/susținere în comentariu (ex: "Guvernul", "Președintele", "Politicienii", "Imigranții", "Un anumit partid" etc.)

2. stance: Poziția comentariului față de target - poate fi "pro" (susținere), "against" (împotrivă), "neutral" (neutru), "ambivalent" (mixt/contradictoriu, trist/apreciativ)

3. sentiment: Emoția predominantă exprimată - poate fi "positive", "negative", "neutral", "angry", "frustrated", "hopeful", "sarcastic"

4. tone: Tonul general al comentariului - poate fi "formal", "informal", "aggressive", "respectful", "humorous", "sarcastic", "urgent", "fatalist"

5. topic: Subiectul principal discutat - poate fi "economy", "healthcare", "education", "immigration", "corruption", "foreign policy", "environment", "social rights", "security", "elections" etc.

6. interpretation_problem: Probleme de interpretare identificate - poate fi "ambiguity" (ambiguitate), "sarcasm" (sarcasm greu de detectat), "missing_context" (lipsă context), "contradiction" (contradicții interne), "none" (fără probleme)

Important:
- Analizează DOAR textul comentariului furnizat
- Dacă un element nu poate fi determinat clar, folosește "unknown" pentru stance/sentiment/tone/topic și "unclear" pentru target/interpretation_problem
- Nu adăuga interpretări personale sau informații externe
- Răspunde DOAR cu JSON valid, fără text suplimentar înainte sau după

Returnează JSON valid cu exact aceste chei:
target, stance, sentiment, tone, topic, interpretation_problem

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [11]:
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [12]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [13]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = sample_df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,"```json\n{\n ""target"": ""Președintele Călin Ge..."
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,"```json\n{\n ""target"": ""Călin Georgescu"",\n ..."
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,"```json\n{\n ""target"": ""Autoritățile"",\n ""st..."
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,"```json\n{\n ""target"": ""Persoanele care se af..."
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,"```json\n{\n ""target"": ""Sistemul de protecție..."
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,"```json\n{\n ""target"": ""Judecătorii/Sistemul ..."
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...","```json\n{\n ""target"": ""Noua Ordine Mondială,..."
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...","```json\n{\n ""target"": ""Un politician corupt""..."
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...","```json\n{\n ""target"": ""Crin Antonescu"",\n ""..."
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,"```json\n{\n ""target"": ""Firme din Galați"",\n ..."


# 9. Verificam rezultatele

In [14]:
results_df.model_output[0]

'```json\n{\n  "target": "Președintele Călin Georgescu",\n  "stance": "against",\n  "sentiment": "sarcastic",\n  "tone": "sarcastic",\n  "topic": "elections",\n  "interpretation_problem": "none"\n}\n```'

In [15]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [16]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,Președintele Călin Georgescu,against,sarcastic,sarcastic,elections,none,
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,Călin Georgescu,pro,positive,respectful,politics,none,
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,Autoritățile,against,angry,aggressive,corruption,none,
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,Persoanele care se află sau trec prin zona res...,neutral,frustrated,fatalist,security,unclear,
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,"Sistemul de protecție a copilului, societatea",against,negative,sad,social rights,none,
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,Judecătorii/Sistemul judiciar,against,frustrated,aggressive,corruption,none,
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...","Noua Ordine Mondială, Agenda 2030",against,angry,aggressive,security,none,
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...",Un politician corupt,against,angry,aggressive,corruption,none,
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...",Crin Antonescu,against,sarcastic,sarcastic,elections,none,
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,Firme din Galați,against,negative,informal,corruption,missing_context,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [17]:
results_df.to_csv('tema1_output.csv', index=False, encoding='utf-8-sig')

Where did the prompt work well?
Where did it fail?
Did it confuse sentiment with stance?
Did sarcasm, ambiguity, or multiple targets create problems?
What would I change in the next version of the prompt?

In mare parte dintre instante a mapat corect campurile, a functionat mai bine dupa ce am ajustat promptul, mentionand sa ia context din titlul stirii.

Probleme identificate:
- nu preia context si din titlul stirii, ci doar priveste comentariul. Un caz in care titlul mentiona pe Crin Antonescu, target a fost identificat ca fiind "Un hot corupt", sau in alte cazuri unde persoane erau mentionate nominal in titlu, targetul nu a fost identificat.
- supraestimeaza sarcasmul si unele sentimente.